# Handwritten digit classification with scikit-learn

This notebook reproduces and explains the main workflow from scikit-learn's official **Recognizing hand-written digits** example.

The objective is to understand a complete classical machine-learning classification workflow:

**image data → numerical features → train/test split → model training → prediction → evaluation**

The dataset contains small grayscale images of handwritten digits from **0 to 9**.  
Each image is **8 × 8 pixels**, so each image can be represented using **64 numerical features**.


## 1. Import the required libraries

We first import the libraries needed for the experiment.

- `numpy` will be useful for numerical operations later.
- `matplotlib` is used to visualize digit images and the confusion matrix.
- `datasets` provides the built-in handwritten digits dataset.
- `svm` provides the Support Vector Classifier used in the official example.
- `metrics` provides tools for evaluating the classifier.
- `train_test_split` divides the dataset into training and testing subsets.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn import datasets, metrics, svm
from sklearn.model_selection import train_test_split


## 2. Load and inspect the digits dataset

`load_digits()` loads the dataset directly from scikit-learn, so no external dataset download is required.

We inspect the shapes to understand how the data is stored:

- `digits.images` stores the original **8 × 8 images**.
- `digits.data` stores the same images already flattened into **64 numerical features**.
- `digits.target` stores the correct digit label for each image.


In [ ]:
digits = datasets.load_digits()

print("Images shape:", digits.images.shape)
print("Data shape:", digits.data.shape)
print("Target shape:", digits.target.shape)
print("Target names:", digits.target_names)


### Interpreting the dataset structure

The expected shapes are approximately:

```text
Images shape: (1797, 8, 8)
Data shape:   (1797, 64)
Target shape: (1797,)
```

This means there are **1797 samples**.

For each sample:

```text
8 × 8 image
    ↓
64 pixel values
    ↓
64 numerical features
```

The target is one of the ten digit classes: `0` through `9`.


## 3. Visualize a few training samples

Before training a model, it is useful to look at the raw data.

The cell below displays four handwritten digit images together with their true labels.  
This helps connect the numerical pixel data with the actual visual pattern represented by each sample.


In [ ]:
_, axes = plt.subplots(nrows=1, ncols=4, figsize=(10, 3))

for ax, image, label in zip(axes, digits.images, digits.target):
    ax.set_axis_off()
    ax.imshow(image, cmap=plt.cm.gray_r, interpolation="nearest")
    ax.set_title(f"Training: {label}")

plt.tight_layout()
plt.show()


## 4. Convert each image into numerical features

Most scikit-learn estimators expect the input feature matrix in the form:

```text
(number of samples, number of features)
```

The original images have the shape:

```text
(1797, 8, 8)
```

We therefore flatten every `8 × 8` image into a vector containing `64` values.

The result becomes:

```text
(1797, 64)
```

Each pixel intensity acts as one numerical feature for the classifier.


In [ ]:
n_samples = len(digits.images)

data = digits.images.reshape((n_samples, -1))

print("Original image data shape:", digits.images.shape)
print("Flattened feature matrix shape:", data.shape)


### Check against `digits.data`

scikit-learn already provides the flattened representation in `digits.data`.

The following check confirms that our manually reshaped version contains the same values.


In [ ]:
print("Are the reshaped images and digits.data identical?")
print(np.array_equal(data, digits.data))


## 5. Create the Support Vector Classifier

The official example uses a **Support Vector Classifier (SVC)**.

Here we create the model with:

```python
gamma=0.001
```

`gamma` influences how strongly individual training samples affect the decision boundary.

For this first experiment, we keep the parameter identical to the official example rather than tuning it.


In [ ]:
clf = svm.SVC(gamma=0.001)

print(clf)


## 6. Split the dataset into training and test data

A model must be evaluated on data that was not used during training.

The official example uses:

```text
50% training data
50% testing data
```

and sets:

```python
shuffle=False
```

This means the samples remain in their original order.

Later, we can compare this with a more typical split such as `80/20` with stratification.  
For now, we stay close to the official scikit-learn implementation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data,
    digits.target,
    test_size=0.5,
    shuffle=False
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)


## 7. Train the classifier

Training happens when we call:

```python
clf.fit(X_train, y_train)
```

Here:

- `X_train` contains the pixel features.
- `y_train` contains the correct digit labels.

The classifier learns patterns in the pixel values that help distinguish digits such as `0`, `1`, `2`, ..., `9`.


In [ ]:
clf.fit(X_train, y_train)

print("Model training complete.")


## 8. Predict the test data

After training, the classifier is applied to the unseen test features.

`predict()` returns one predicted class for every test image.

We also display a few predictions alongside their actual labels.


In [ ]:
predicted = clf.predict(X_test)

print("First 20 predicted labels:")
print(predicted[:20])

print("\nFirst 20 actual labels:")
print(y_test[:20])


## 9. Visualize example predictions

The flattened test samples contain 64 values each.

For visualization, each sample is reshaped back into an `8 × 8` image.

The title displays both:

- `P` = predicted label
- `A` = actual label


In [ ]:
_, axes = plt.subplots(nrows=1, ncols=4, figsize=(10, 3))

for ax, image, prediction, actual in zip(
    axes, X_test, predicted, y_test
):
    ax.set_axis_off()

    image = image.reshape(8, 8)

    ax.imshow(
        image,
        cmap=plt.cm.gray_r,
        interpolation="nearest"
    )

    ax.set_title(f"P: {prediction} | A: {actual}")

plt.tight_layout()
plt.show()


## 10. Evaluate the classifier

A single accuracy value does not tell us everything about a multiclass classifier.

The classification report provides several metrics for every digit:

- **Precision** — when the model predicts a particular digit, how often is it correct?
- **Recall** — of all actual examples of that digit, how many does the model identify correctly?
- **F1-score** — balances precision and recall.
- **Support** — number of test samples belonging to that class.

The report also includes overall accuracy and aggregate averages.


In [ ]:
print(
    f"Classification report for classifier {clf}:\n"
    f"{metrics.classification_report(y_test, predicted)}"
)


## 11. Calculate overall accuracy

Accuracy is the fraction of test samples classified correctly.

It is useful as a high-level summary, but the confusion matrix and per-class metrics give more detail about the model's mistakes.


In [ ]:
accuracy = metrics.accuracy_score(y_test, predicted)

print(f"Test accuracy: {accuracy:.4f}")
print(f"Test accuracy: {accuracy * 100:.2f}%")


## 12. Create the confusion matrix

The confusion matrix shows how predictions are distributed across the ten digit classes.

- Rows correspond to **actual digits**.
- Columns correspond to **predicted digits**.
- Values on the main diagonal represent correct predictions.
- Values away from the diagonal represent classification mistakes.

This makes it possible to identify which digits are most often confused with one another.


In [ ]:
disp = metrics.ConfusionMatrixDisplay.from_predictions(
    y_test,
    predicted
)

disp.figure_.suptitle("Confusion Matrix")

plt.show()

print("Confusion matrix:")
print(disp.confusion_matrix)


## 13. Rebuild the classification report from the confusion matrix

This section comes from the official scikit-learn example.

A confusion matrix contains the number of occurrences for every:

```text
actual class → predicted class
```

pair.

The code below reconstructs lists of actual and predicted labels using those counts.  
Generating the classification report again demonstrates that the confusion matrix contains enough information to recover the same classification metrics.

This step is useful conceptually, but it is **not required for training or using the classifier**.


In [ ]:
y_true = []
y_pred = []

cm = disp.confusion_matrix

for gt in range(len(cm)):
    for pred in range(len(cm)):
        y_true += [gt] * cm[gt][pred]
        y_pred += [pred] * cm[gt][pred]

print(
    "Classification report rebuilt from confusion matrix:\n"
    f"{metrics.classification_report(y_true, y_pred)}"
)


## Summary

In this notebook we completed a full classical machine-learning classification workflow:

```text
handwritten image
      ↓
8 × 8 pixel matrix
      ↓
64 numerical features
      ↓
train/test split
      ↓
Support Vector Classifier
      ↓
model training
      ↓
prediction on unseen samples
      ↓
accuracy + precision + recall + F1
      ↓
confusion matrix
```

The next useful step is to inspect the model's **misclassified images** and determine which digit pairs are responsible for the most errors.

After that, we can compare this SVM baseline with a second model such as **Logistic Regression** while keeping the rest of the workflow unchanged.
